# Manual LLM-judge evaluation of open-ended answers

Replaces `ragas-open-ended-evaluation.ipynb`. RAGAS's `answer_relevancy` metric
internally generates a synthetic question from the answer and embeds it to compute
cosine similarity against the real question -- when the judge model (`tencent/hy4-flash`
via OpenRouter) occasionally fails to produce that synthetic question, RAGAS still
passes the invalid value straight to the embeddings endpoint, which OpenRouter
rejects with a 422. This happened on a large fraction of rows regardless of
`strictness`, burning real money on retries without producing usable scores.

This notebook scores `faithfulness`, `answer_relevancy`, and `answer_correctness`
with a single deterministic (`temperature=0`, JSON-mode) chat completion per row,
calling OpenRouter directly -- no RAGAS, no LangChain, no embeddings call. One call
per row means lower cost and no multi-sample/embedding failure mode.

In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().parent
load_dotenv(REPO_ROOT / ".env")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

REASONING_MODEL_FILTER = "gemma-4-26b-a4b-it"
EMBEDDING_MODEL_FILTER = "qwen06b"
JUDGE_MODEL = "z-ai/glm-5.3-flash"
MAX_WORKERS = 5
MAX_RETRIES = 2

METRIC_COLUMNS = ["faithfulness", "answer_relevancy", "answer_correctness"]

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)


def connect():
    conn_str = os.getenv("PG_CONNECTION_STRING")
    if conn_str:
        return psycopg2.connect(conn_str)
    return psycopg2.connect(host="localhost", dbname="docs", user="postgres", password="password")

/Users/fahmisyaifudin/project/academic/odoo-docs-graph/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load open-ended rows with their requirements document(s) and ground truth

In [2]:
QUERY = """
    SELECT
        ar.id AS answer_result_id,
        ar.question_id,
        ar.method,
        ar.embedding_model,
        ar.llm_reasoning_model,
        q.question,
        q.answer_text AS ground_truth,
        ar.result AS answer,
        array_agg(d.document) AS contexts
    FROM answer_result ar
    JOIN question q ON q.id = ar.question_id
    JOIN question_documents qd ON qd.question_id = q.id
    JOIN documents d ON d.id = qd.document_id
    WHERE q.is_open_ended = true
      AND ar.llm_reasoning_model = %(reasoning_model)s
      AND ar.embedding_model = %(embedding_model)s
      AND ar.result IS NOT NULL
      AND q.answer_text IS NOT NULL
      AND d.document IS NOT NULL
    GROUP BY ar.id, ar.question_id, ar.method, ar.embedding_model, ar.llm_reasoning_model,
             q.question, q.answer_text, ar.result
"""

with connect() as conn:
    df = pd.read_sql(
        QUERY, conn,
        params={"reasoning_model": REASONING_MODEL_FILTER, "embedding_model": EMBEDDING_MODEL_FILTER},
    )

print(f"{len(df)} row(s) to evaluate")
df.head()

/var/folders/6s/4n20n38x5y571387phy8c5lc0000gn/T/ipykernel_45730/3658828869.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


599 row(s) to evaluate


,answer_result_id,question_id,method,embedding_model,llm_reasoning_model,question,ground_truth,answer,contexts
0,58c3c7b4-b2d5-4858-9a83-2934e4d096fe,4ce99683-0029-4b92-a6d2-4e33b0fcef80,agent,qwen06b,gemma-4-26b-a4b-it,What is affected if the audit trail retention ...,FR-AUD-06 and BR-08 require retention for at l...,If the audit trail retention period specified ...,[# CCTNS — Standard Requirements Specification...
1,3c8ef03c-3a2e-4bd1-95fd-50ff3ff48914,ba3ee911-f374-4079-8ec8-62ff209a17d8,agent,qwen06b,gemma-4-26b-a4b-it,What is affected if the default customer role ...,FR-ACC-07 sets new accounts to the customer ro...,Changing the default customer role specified i...,[# GAMMA-J Web Store — Standard Requirements S...
2,e56144de-c7c7-46b3-8f76-cfdff389cca2,d8097238-f32b-4c53-8576-ba60608f1b0b,agent,qwen06b,gemma-4-26b-a4b-it,What changes if identifiers become editable; g...,BR-05 allows all asset fields to be modified e...,Based on the business rules for the UUIS produ...,[# UUIS — Standard Requirements Specification\...
3,808ceffe-b3ec-4348-beb6-f65fb3904b8a,281ed743-49c9-46e7-a877-321a51b28575,pgvector,qwen06b,gemma-4-26b-a4b-it,What changes if daily processing (BR-01) becom...,BR-01 requires daily processing; weekly proces...,If the daily acquisition and processing of ASP...,[# APAF — Standard Requirements Specification\...
4,0fe95ef3-a1b8-4bd0-931b-97f19ec56ba7,a785e81d-62c1-4482-ab0b-b861741c0cde,agent,qwen06b,gemma-4-26b-a4b-it,What changes if the DB2-to-Oracle transfer mov...,Use Case 7 specifies nightly incremental updat...,Moving the DB2-to-Oracle transfer from a night...,[# Water Use Tracking (WUT) System — Standard ...


## Judge prompt and scoring function

One call per row. `temperature=0` for determinism, `response_format=json_object` so
the model is constrained to valid JSON. Each reasoning field is asked for *before*
its score so the model has to justify the number it gives (a cheap stand-in for
RAGAS's claim decomposition, without the extra calls).

In [3]:
SYSTEM_PROMPT = """You are an expert evaluator for RAG (retrieval-augmented generation) systems \
answering requirements-specification questions. You will be given a QUESTION, the retrieved \
CONTEXT (reference requirements documents), the system's ANSWER, and a GROUND_TRUTH reference answer.

Score the ANSWER on three dimensions, each from 0.0 to 1.0:
1. faithfulness: the proportion of claims in the ANSWER that are directly supported by the CONTEXT. \
An unsupported or contradicted claim lowers this score. 1.0 means every claim is grounded in the \
CONTEXT; 0.0 means none are.
2. answer_relevancy: how directly and completely the ANSWER addresses the QUESTION, without \
irrelevant or missing information. 1.0 is fully relevant and complete; 0.0 is off-topic or misses \
the point entirely.
3. answer_correctness: how factually consistent the ANSWER is with the GROUND_TRUTH, considering \
both overlapping facts and contradictions. 1.0 is fully correct; 0.0 is entirely wrong.

Respond with ONLY a JSON object, no markdown, no commentary, with exactly these keys in this order:
{"faithfulness_score": <float 0.0-1.0>, \
"answer_relevancy_score": <float 0.0-1.0>, \
"answer_correctness_score": <float 0.0-1.0>}"""


def build_user_prompt(row) -> str:
    context = "\n\n---\n\n".join(row.contexts)
    return (
        f"QUESTION:\n{row.question}\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"GROUND_TRUTH:\n{row.ground_truth}\n\n"
        f"ANSWER:\n{row.answer}"
    )


def clamp01(value) -> float:
    return max(0.0, min(1.0, float(value)))


def score_row(row) -> dict:
    user_prompt = build_user_prompt(row)
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
            )
            parsed = json.loads(resp.choices[0].message.content)
            return {
                "faithfulness": clamp01(parsed["faithfulness_score"]),
                "answer_relevancy": clamp01(parsed["answer_relevancy_score"]),
                "answer_correctness": clamp01(parsed["answer_correctness_score"]),
            }
        except Exception as e:  # noqa: BLE001 - keep going on malformed output or transient errors
            last_error = e
    print(f"Failed after {MAX_RETRIES + 1} attempt(s) for answer_result_id={row.answer_result_id}: {last_error}")
    return {col: float("nan") for col in METRIC_COLUMNS}

## Evaluate

In [4]:
scores = [None] * len(df)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(score_row, row): idx for idx, row in enumerate(df.itertuples())}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Scoring"):
        idx = futures[future]
        scores[idx] = future.result()

results_df = pd.concat(
    [
        df[["answer_result_id", "question_id", "method", "embedding_model", "llm_reasoning_model"]].reset_index(drop=True),
        pd.DataFrame(scores),
    ],
    axis=1,
)

nan_count = results_df[METRIC_COLUMNS].isna().any(axis=1).sum()
print(f"{nan_count} row(s) failed to score")
results_df.head()

Scoring:  62%|██████▏   | 374/599 [12:38<08:59,  2.40s/it]

Failed after 3 attempt(s) for answer_result_id=37f2f724-7b11-4b4a-8183-c2fe6a8c4675: Error code: 429 - {'error': {'code': 'AccountQuotaExceeded', 'message': 'You have exceeded the 5-hour usage quota. It will reset at 2026-11-23 00:29:41 +0800 CST. We recommend upgrading your plan for more quota, or waiting for the reset. Request id: 0217953532333055d6d8f07cf2a752c74db4b42ef6a643e804bf0', 'param': '', 'type': 'TooManyRequests'}}


Scoring:  63%|██████▎   | 375/599 [12:41<09:48,  2.63s/it]

Failed after 3 attempt(s) for answer_result_id=4e5a6eb4-5807-45d0-a89d-7e0bd04e56b6: Error code: 429 - {'error': {'code': 'AccountQuotaExceeded', 'message': 'You have exceeded the 5-hour usage quota. It will reset at 2026-11-23 00:29:41 +0800 CST. We recommend upgrading your plan for more quota, or waiting for the reset. Request id: 021795353236505651bcb683a664fec8e1b815c68d8e2dee6b591', 'param': '', 'type': 'TooManyRequests'}}


Scoring:  63%|██████▎   | 377/599 [12:43<06:29,  1.76s/it]

Failed after 3 attempt(s) for answer_result_id=80a024ca-3323-442b-93c5-9f04e2e05dcf: Error code: 429 - {'error': {'code': 'AccountQuotaExceeded', 'message': 'You have exceeded the 5-hour usage quota. It will reset at 2026-11-23 00:29:41 +0800 CST. We recommend upgrading your plan for more quota, or waiting for the reset. Request id: 0217953532382425d6d8f07cf2a752c74db4b42ef6a643e785079', 'param': '', 'type': 'TooManyRequests'}}


... (383 outputs left out)

Failed after 3 attempt(s) for answer_result_id=fb11372a-5ad9-4d60-95bc-e82159d5139f: Error code: 429 - {'error': {'code': 'AccountRateLimitExceeded', 'message': 'Requests are too frequent. Please reduce your request frequency, wait a short moment, and retry your request. Request id: 0217953534847735d6d8f07cf2a752c74db4b42ef6a643ef00d27', 'param': '', 'type': 'TooManyRequests'}}


Scoring: 100%|██████████| 599/599 [16:51<00:00,  1.69s/it]

Failed after 3 attempt(s) for answer_result_id=1f210edd-2c40-496a-99b2-752954898405: Error code: 429 - {'error': {'code': 'AccountQuotaExceeded', 'message': 'You have exceeded the 5-hour usage quota. It will reset at 2026-11-23 00:29:41 +0800 CST. We recommend upgrading your plan for more quota, or waiting for the reset. Request id: 021795353486210651bcb683a664fec8e1b815c68d8e2de29d020', 'param': '', 'type': 'TooManyRequests'}}
223 row(s) failed to score


,answer_result_id,question_id,method,embedding_model,llm_reasoning_model,faithfulness,answer_relevancy,answer_correctness
0,58c3c7b4-b2d5-4858-9a83-2934e4d096fe,4ce99683-0029-4b92-a6d2-4e33b0fcef80,agent,qwen06b,gemma-4-26b-a4b-it,0.9,0.65,0.60
1,3c8ef03c-3a2e-4bd1-95fd-50ff3ff48914,ba3ee911-f374-4079-8ec8-62ff209a17d8,agent,qwen06b,gemma-4-26b-a4b-it,0.1,0.10,0.00
2,e56144de-c7c7-46b3-8f76-cfdff389cca2,d8097238-f32b-4c53-8576-ba60608f1b0b,agent,qwen06b,gemma-4-26b-a4b-it,1.0,0.40,0.40
3,808ceffe-b3ec-4348-beb6-f65fb3904b8a,281ed743-49c9-46e7-a877-321a51b28575,pgvector,qwen06b,gemma-4-26b-a4b-it,1.0,0.40,0.40
4,0fe95ef3-a1b8-4bd0-931b-97f19ec56ba7,a785e81d-62c1-4482-ab0b-b861741c0cde,agent,qwen06b,gemma-4-26b-a4b-it,1.0,0.70,0.75


## Aggregate scores

In [5]:
mean_by_method = results_df.groupby("method")[METRIC_COLUMNS].mean()
mean_by_embedding_model = results_df.groupby("embedding_model")[METRIC_COLUMNS].mean()

print("Mean scores by method:")
display(mean_by_method)

print("\nMean scores by embedding_model:")
display(mean_by_embedding_model)

mean_by_method.to_csv("llm_judge_mean_scores_by_method.csv")
mean_by_embedding_model.to_csv("llm_judge_mean_scores_by_embedding_model.csv")
print("\nSaved llm_judge_mean_scores_by_method.csv and llm_judge_mean_scores_by_embedding_model.csv")

Mean scores by method:


,faithfulness,answer_relevancy,answer_correctness
method,,,
agent,0.749231,0.597308,0.542692
neo4j,0.811062,0.619027,0.560619
pgvector,0.877669,0.675564,0.652857



Mean scores by embedding_model:


,faithfulness,answer_relevancy,answer_correctness
embedding_model,,,
qwen06b,0.813245,0.631516,0.587048



Saved llm_judge_mean_scores_by_method.csv and llm_judge_mean_scores_by_embedding_model.csv


## Write per-question scores to `ragas_evaluation`

`ragas_evaluation` has no unique constraint on `answer_result_id`. If this reasoning
model/embedding_model combination was already inserted by a RAGAS run, delete those
rows first so this replaces them rather than adding duplicates.

In [6]:
DELETE_SQL = "DELETE FROM ragas_evaluation WHERE llm_reasoning_model = %s AND embedding_model = %s"
INSERT_SQL = """
    INSERT INTO ragas_evaluation (
        answer_result_id, question_id, method, embedding_model, llm_reasoning_model,
        faithfulness, answer_relevancy, answer_correctness, judge_model
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

conn = connect()
with conn.cursor() as cur:
    cur.execute(DELETE_SQL, (REASONING_MODEL_FILTER, EMBEDDING_MODEL_FILTER))
    deleted = cur.rowcount
    inserted = 0
    for row in results_df.itertuples():
        cur.execute(
            INSERT_SQL,
            (
                row.answer_result_id,
                row.question_id,
                row.method,
                row.embedding_model,
                row.llm_reasoning_model,
                float(row.faithfulness),
                float(row.answer_relevancy),
                float(row.answer_correctness),
                JUDGE_MODEL,
            ),
        )
        inserted += cur.rowcount
conn.commit()
conn.close()
print(f"Deleted {deleted} old row(s), inserted {inserted} row(s) into ragas_evaluation")

Deleted 599 old row(s), inserted 599 row(s) into ragas_evaluation


## Sanity check

In [7]:
# with connect() as conn:
#     check_df = pd.read_sql(
#         """
#         SELECT re.*, q.question
#         FROM ragas_evaluation re
#         JOIN question q ON q.id = re.question_id
#         ORDER BY re.evaluated_at DESC
#         LIMIT 10
#         """,
#         conn,
#     )
# check_df